In [ ]:
df = pd.read_csv("Sepsis Prediction Dataset.csv")
print("Total Rows: ",df.shape[0])
print("Total Columns: ", df.shape[1])

print("Number of Duplicate Rows: ", df.duplicated().sum())

print(f"Number of Unique Patients: {df['Patient_ID'].nunique()}")

print(f"Rows with missing Patient ID: {df['Patient_ID'].isna().sum()}")


import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout, Concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU(s) detected and memory growth enabled.")
    except Exception as e:
        print("GPU config error:", e)
else:
    print("No GPU found. Running on CPU.")


CSV_PATH = "Sepsis Prediction Dataset.csv"
TARGET = "SepsisLabel"
GROUP = "Patient_ID"
HOUR_COL = "Hour"

N_STEPS = 12
TEST_SIZE = 0.20
VAL_SIZE = 0.10
RANDOM_STATE = 42
TARGET_POS_RATIO = 0.20
BATCH_SIZE = 128
EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 1e-3
LSTM_UNITS = 64
GRU_UNITS = 64
DROPOUT = 0.3
PREDICTION_WINDOW = 6   # Predict 6 hours ahead

# Helper Functions
def create_sequences_per_patient(df, feature_cols,
                                 group_col=GROUP,
                                 hour_col=HOUR_COL,
                                 n_steps=N_STEPS,
                                 prediction_window=6):

    X_list, y_list, groups_list = [], [], []
    df_sorted = df.sort_values([group_col, hour_col])

    from collections import defaultdict
    map_idx = defaultdict(list)

    for idx, pid in enumerate(df_sorted[group_col].values):
        map_idx[pid].append(idx)

    features = df_sorted[feature_cols].values
    labels = df_sorted[TARGET].values

    for pid, positions in map_idx.items():

        for i in range(len(positions)):

            current_pos = positions[i]

            # --- Build sequence ---
            start_idx = max(0, i - (n_steps - 1))
            window_positions = positions[start_idx:i + 1]

            window_feat = features[window_positions]

            # Pad if needed
            if window_feat.shape[0] < n_steps:
                pad = np.tile(window_feat[0:1], (n_steps - window_feat.shape[0], 1))
                window_feat = np.vstack([pad, window_feat])

            # --- FUTURE LABEL ---
            future_positions = positions[i+1:i+1+prediction_window]

            if len(future_positions) > 0:
                future_labels = labels[future_positions]
                y = 1 if np.any(future_labels == 1) else 0
            else:
                y = 0

            X_list.append(window_feat)
            y_list.append(y)
            groups_list.append(pid)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32), np.array(groups_list)

def upsample_positives(X, y, groups, target_pos_ratio=TARGET_POS_RATIO, random_state=RANDOM_STATE):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    if len(pos_idx) == 0: return X, y, groups

    current_pos_ratio = len(pos_idx) / (len(pos_idx) + len(neg_idx))
    if current_pos_ratio >= target_pos_ratio: return X, y, groups

    total_needed = int((len(neg_idx) / (1 - target_pos_ratio)) - len(neg_idx))
    n_samples = max(0, total_needed - len(pos_idx))

    if n_samples > 0:
        X_pos_extra = resample(X[pos_idx], replace=True, n_samples=n_samples, random_state=random_state)
        y_pos_extra = np.ones(n_samples, dtype=int)
        groups_pos_extra = resample(groups[pos_idx], replace=True, n_samples=n_samples, random_state=random_state)
        X_up = np.concatenate([X, X_pos_extra], axis=0)
        y_up = np.concatenate([y, y_pos_extra], axis=0)
        groups_up = np.concatenate([groups, groups_pos_extra], axis=0)
    else:
        X_up, y_up, groups_up = X, y, groups

    perm = np.random.RandomState(random_state).permutation(len(y_up))
    return X_up[perm], y_up[perm], groups_up[perm]

# --- EXECUTION ---
print("Loading and Processing Data...")
df = pd.read_csv(CSV_PATH)
if 'Unnamed: 0' in df.columns: df = df.drop(columns=['Unnamed: 0'])

# Handling Missing Values
missing_pct = df.isnull().mean()
drop_cols = missing_pct[missing_pct > 0.95].index.tolist()
if drop_cols: df = df.drop(columns=drop_cols)

df = df.sort_values([GROUP, HOUR_COL]).reset_index(drop=True)
df = df.groupby(GROUP).apply(lambda g: g.ffill().bfill()).reset_index(drop=True)

# Imputation
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude = [TARGET, GROUP, HOUR_COL]
feat_num_cols = [c for c in num_cols if c not in exclude]
imputer = SimpleImputer(strategy='median')
df[feat_num_cols] = imputer.fit_transform(df[feat_num_cols])
feature_cols = feat_num_cols.copy()

# ---- ADD TREND FEATURES ----
print("Creating trend features...")

for col in feature_cols:
    df[col + "_diff"] = df.groupby(GROUP)[col].diff().fillna(0)

print("Trend features created.")


# Splitting
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df[GROUP].values))
df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=RANDOM_STATE)
train_idx2, val_idx2 = next(gss2.split(df_train, groups=df_train[GROUP].values))
df_tr = df_train.iloc[train_idx2].reset_index(drop=True)
df_val = df_train.iloc[val_idx2].reset_index(drop=True)

# Scaling
scaler = StandardScaler()
df_tr[feature_cols] = scaler.fit_transform(df_tr[feature_cols])
df_val[feature_cols] = scaler.transform(df_val[feature_cols])
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

# Create Sequences
print("Generating sequences...")
X_tr, y_tr, groups_tr = create_sequences_per_patient(
    df_tr, feature_cols, prediction_window=PREDICTION_WINDOW
)

X_val, y_val, groups_val = create_sequences_per_patient(
    df_val, feature_cols, prediction_window=PREDICTION_WINDOW
)

X_test, y_test, groups_test = create_sequences_per_patient(
    df_test, feature_cols, prediction_window=PREDICTION_WINDOW
)

# Upsampling
# X_tr_up, y_tr_up, groups_tr_up = upsample_positives(X_tr, y_tr, groups_tr)
X_tr_up, y_tr_up = X_tr, y_tr

print("Data ready. Training shape:", X_tr_up.shape)

from sklearn.linear_model import SGDClassifier

# Flatten 3D sequences to 2D for traditional ML
print("Training Traditional Baselines (Fast Version)...")
X_tr_flat = X_tr_up.reshape(X_tr_up.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

results_dict = {}

# 1. Logistic Regression (Approximation using SGD)
# SGDClassifier with loss='log_loss' is mathematically identical to Logistic Regression
# but trains much faster on large datasets.
print("  - Training Logistic Regression (SGD)...")
lr = SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42, n_jobs=-1)
lr.fit(X_tr_flat, y_tr_up)
lr_probs = lr.predict_proba(X_test_flat)[:, 1]
lr_pred = (lr_probs >= 0.5).astype(int)

results_dict['Logistic Regression'] = {
    'Accuracy': accuracy_score(y_test, lr_pred),
    'F1': f1_score(y_test, lr_pred),
    'AUROC': roc_auc_score(y_test, lr_probs)
}

# 2. Random Forest (Optimized for speed)
# We use max_samples=0.1 to train each tree on only 10% of the data
print("  - Training Random Forest...")
rf = RandomForestClassifier(n_estimators=30, max_samples=0.1, class_weight='balanced', n_jobs=-1, random_state=42)
rf.fit(X_tr_flat, y_tr_up)
rf_probs = rf.predict_proba(X_test_flat)[:, 1]
rf_pred = (rf_probs >= 0.5).astype(int)

results_dict['Random Forest'] = {
    'Accuracy': accuracy_score(y_test, rf_pred),
    'F1': f1_score(y_test, rf_pred),
    'AUROC': roc_auc_score(y_test, rf_probs)
}

print("Baselines Done.")

def focal_loss(gamma=2., alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true_f = tf.cast(y_true, tf.float32)
        eps = 1e-8
        p_t = y_true_f * y_pred + (1 - y_true_f) * (1 - y_pred)
        alpha_factor = y_true_f * alpha + (1 - y_true_f) * (1 - alpha)
        modulating_factor = tf.pow(1.0 - p_t, gamma)
        loss = -alpha_factor * modulating_factor * tf.math.log(p_t + eps)
        return tf.reduce_mean(loss)
    return loss_fn

def make_dataset(X, y, batch_size=BATCH_SIZE, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(buffer_size=min(100000, len(y)))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_tr_up, y_tr_up, shuffle=True)
val_ds = make_dataset(X_val, y_val, shuffle=False)
test_ds = make_dataset(X_test, y_test, shuffle=False)

# pos = (y_tr_up == 1).sum()
# neg = (y_tr_up == 0).sum()
# class_weight = {0: 1.0, 1: float(max(1.0, (neg / (pos + 1e-9))))}

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_tr)
weights = compute_class_weight(class_weight='balanced',
                               classes=classes,
                               y=y_tr)

class_weight = {0: weights[0], 1: weights[1]}

input_shape = X_tr_up.shape[1:]

# Helper to train and evaluate DL models
def train_evaluate_dl(model, name):
    print(f"Training {name}...")
    callbacks = [EarlyStopping(monitor='val_recall', mode='max', patience=PATIENCE, restore_best_weights=True)]
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, class_weight=class_weight, verbose=0)

    # Predict
    probs = model.predict(test_ds).ravel()
    preds = (probs >= 0.5).astype(int) # Using 0.5 for comparison parity

    return {
        'Accuracy': accuracy_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'AUROC': roc_auc_score(y_test, probs)
    }

# 3. LSTM Only
# inp = Input(shape=input_shape)
# x = LSTM(64, return_sequences=False)(inp)
# x = Dropout(0.3)(x)
# x = BatchNormalization()(x)
# x = Dense(64, activation='relu')(x)
# x = Dropout(0.2)(x)
# out = Dense(1, activation='sigmoid')(x)
# lstm_model = Model(inputs=inp, outputs=out)
# lstm_model.compile(optimizer=Adam(LEARNING_RATE), loss=focal_loss(), metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])

# results_dict['LSTM Only'] = train_evaluate_dl(lstm_model, "LSTM Only")

# 4. GRU Only
inp = Input(shape=input_shape)
x = GRU(64, return_sequences=False)(inp)
x = Dropout(0.3)(x)
x = BatchNormalization()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
out = Dense(1, activation='sigmoid')(x)
gru_model = Model(inputs=inp, outputs=out)
gru_model.compile(optimizer=Adam(LEARNING_RATE), loss=focal_loss(), metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])

val_probs = gru_model.predict(val_ds).ravel()

best_thresh = 0.5
best_f1 = -1

for th in np.linspace(0.05, 0.9, 36):
    preds = (val_probs >= th).astype(int)
    f1 = f1_score(y_val, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = th

print("Best Threshold:", best_thresh)


results_dict['GRU Only'] = train_evaluate_dl(gru_model, "GRU Only")

# print("Training Proposed Hybrid Model...")
# inp = Input(shape=input_shape, name="seq_in")
# x1 = LSTM(LSTM_UNITS, return_sequences=False)(inp)
# x1 = Dropout(DROPOUT)(x1)
# x2 = GRU(GRU_UNITS, return_sequences=False)(inp)
# x2 = Dropout(DROPOUT)(x2)
# x = Concatenate()([x1, x2])
# x = BatchNormalization()(x)
# x = Dense(64, activation='relu')(x)
# x = Dropout(0.2)(x)
# out = Dense(1, activation='sigmoid')(x)

# hybrid_model = Model(inputs=inp, outputs=out)
# hybrid_model.compile(optimizer=Adam(LEARNING_RATE), loss=focal_loss(), metrics=['accuracy', tf.keras.metrics.Recall(name='recall')])

# # Using your original callback logic
# callbacks = [
#     EarlyStopping(monitor='val_recall', mode='max', patience=PATIENCE, restore_best_weights=True, verbose=1),
#     ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
# ]

# history = hybrid_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, class_weight=class_weight, verbose=1)

# # Tune Threshold (Your original logic)
# val_probs = hybrid_model.predict(val_ds).ravel()
# best_thresh = 0.5
# best_f1 = -1
# for th in np.linspace(0.05, 0.9, 36):
#     y_pred = (val_probs >= th).astype(int)
#     f1 = f1_score(y_val, y_pred, zero_division=0)
#     if f1 > best_f1:
#         best_f1 = f1
#         best_thresh = th

# # Final Evaluation
# test_probs = hybrid_model.predict(test_ds).ravel()
# test_pred = (test_probs >= best_thresh).astype(int)

# results_dict['Hybrid (Proposed)'] = {
#     'Accuracy': accuracy_score(y_test, test_pred),
#     'F1': f1_score(y_test, test_pred),
#     'AUROC': roc_auc_score(y_test, test_probs)
# }

# # Print Final Comparison
# print("\n--- FINAL COMPARATIVE RESULTS ---")
# results_df = pd.DataFrame(results_dict).T
# print(results_df)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_patient_live(patient_id, df, model, scaler, feature_cols,
                          group_col="Patient_ID",
                          hour_col="Hour",
                          threshold=0.5,
                          n_steps=6):
    """
    Simulates real-time sepsis risk prediction for a single patient.
    """

    # Select patient data
    patient_df = df[df[group_col] == patient_id].copy()

    if patient_df.empty:
        print("Patient not found.")
        return

    patient_df = patient_df.sort_values(hour_col)

    history_buffer = []
    risk_scores = []
    hours = []
    true_labels = []

    print(f"\n--- Starting Simulation for Patient {patient_id} ---\n")

    for _, row in patient_df.iterrows():

        hour = row[hour_col]
        true_label = row["SepsisLabel"]

        # Extract features for this hour
        new_data = row[feature_cols].values.reshape(1, -1)

        # Scale using trained scaler
        new_data_scaled = scaler.transform(new_data)

        # Add to rolling buffer
        history_buffer.append(new_data_scaled[0])

        # Keep only last n_steps
        if len(history_buffer) > n_steps:
            history_buffer.pop(0)

        # Only predict once buffer is full
        if len(history_buffer) == n_steps:

            X_input = np.array(history_buffer).reshape(1, n_steps, -1)
            risk = model.predict(X_input, verbose=0)[0][0]

            risk_scores.append(risk)
            hours.append(hour)
            true_labels.append(true_label)

            status = "⚠️ ALERT" if risk >= threshold else "OK"

            print(f"Hour {int(hour):3d} | Risk: {risk:.3f} | True: {true_label} | {status}")

    print("\n--- Simulation Complete ---\n")

    # Plot results
    if len(risk_scores) > 0:
        plt.figure(figsize=(12,6))
        plt.plot(hours, risk_scores, label="Predicted Risk")
        plt.axhline(threshold, linestyle='--', label="Alert Threshold")
        plt.scatter(hours, true_labels, marker='x', label="True Sepsis Label")
        plt.title(f"Live Sepsis Risk Simulation (Patient {patient_id})")
        plt.xlabel("Hour")
        plt.ylabel("Risk Probability")
        plt.legend()
        plt.show()

    return pd.DataFrame({
        "Hour": hours,
        "RiskScore": risk_scores,
        "TrueLabel": true_labels
    })

# Find a patient who actually develops sepsis
septic_patients = df_test[df_test["SepsisLabel"] == 1]["Patient_ID"].unique()

selected_patient = septic_patients[0]  # or choose manually

simulation_results = simulate_patient_live(
    patient_id=selected_patient,
    df=df_test,
    model=gru_model,
    scaler=scaler,
    feature_cols=feature_cols,
    threshold=0.5,
    n_steps=12
)
